In [1]:
import warnings
import s3fs
import xarray
import shutil
import icechunk
import xarray_subset_grid as xsg
from obstore.store import from_url
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr.parsers import HDFParser
from virtualizarr import open_virtual_dataset
import re
import fsspec
import datetime
import zarr

icechunk_bucket = "nextgen-dmac-cloud-ingest-cmorse-dev"
icechunk_prefix = "sscofs/sscofs-icechunk-best"
warnings.filterwarnings("ignore", category=UserWarning)
fs = s3fs.S3FileSystem(anon=True)  


In [2]:

bucket = "s3://noaa-nos-ofs-pds/"
registry = ObjectStoreRegistry({bucket: from_url(bucket, region="us-east-1", skip_signature=True)})
parser = HDFParser(drop_variables=["siglay","siglev","Times"])

In [3]:

def parse_nos_model_run_datestamp_offset(key: str) -> (str, int):
    model_hour, model_date, offset = re.search(r'.t(\d{2})z.(\d{8}).*.[f|n](\d{3})', key).groups()
    model_date = datetime.datetime.strptime(f'{model_date}T{model_hour}', '%Y%m%dT%H') + datetime.timedelta(hours=int(offset))
    model_date_key = model_date.strftime('%Y%m%dT%H')
    return model_date_key, int(offset)

best_time_series_glob = '*/sscofs.*.fields.*.nc'

fs_read = fsspec.filesystem('s3', anon=True, skip_instance_cache=True)

model_files = fs_read.glob(f's3://noaa-nos-ofs-pds/sscofs/netcdf/2026/11/{best_time_series_glob}')
model_files = sorted(['s3://'+f for f in model_files])

indexes = {}

for f in model_files:
        model_date_key, offset = parse_nos_model_run_datestamp_offset(f)
        if model_date_key not in indexes:
            indexes[model_date_key] = [offset, f]
        else:
            if offset < indexes[model_date_key][0]:
                indexes[model_date_key] = [offset, f]

model_best_files = [x[1] for x in list(indexes.values())]

len(model_best_files)

517

In [4]:
#Test 1 file
coords = ["time", "lat", "lon","latc","lonc"]
vds = open_virtual_dataset(url=model_best_files[0], parser=parser, registry=registry, 
                           loadable_variables=coords)
vds

<xarray.Dataset> Size: 179MB
Dimensions:             (node: 239734, nele: 433410, time: 1, three: 3,
                         maxnode: 11, maxelem: 9, four: 4, siglay: 10,
                         siglev: 11)
Coordinates:
    lon                 (node) float32 959kB 235.9 235.9 235.9 ... 238.0 238.0
    lat                 (node) float32 959kB 44.39 44.39 44.39 ... 45.63 45.64
    lonc                (nele) float32 2MB 235.9 235.9 235.9 ... 238.0 238.0
    latc                (nele) float32 2MB 44.4 44.4 44.4 ... 45.63 45.64 45.64
  * time                (time) datetime64[ns] 8B 2026-11-01T03:00:00
Dimensions without coordinates: node, nele, three, maxnode, maxelem, four,
                                siglay, siglev
Data variables: (12/43)
    nprocs              int32 4B ManifestArray<shape=(), dtype=int32, chunks=()>
    partition           (nele) int32 2MB ManifestArray<shape=(433410,), dtype...
    x                   (node) float32 959kB ManifestArray<shape=(239734,), d...
    y                   (node) float32 959kB ManifestArray<shape=(239734,), d...
    xc                  (nele) float32 2MB ManifestArray<shape=(433410,), dty...
    yc                  (nele) float32 2MB ManifestArray<shape=(433410,), dty...
    ...                  ...
    wet_nodes           (time, node) int32 959kB ManifestArray<shape=(1, 2397...
    wet_cells           (time, nele) int32 2MB ManifestArray<shape=(1, 433410...
    wet_nodes_prev_int  (time, node) int32 959kB ManifestArray<shape=(1, 2397...
    wet_cells_prev_int  (time, nele) int32 2MB ManifestArray<shape=(1, 433410...
    wet_cells_prev_ext  (time, nele) int32 2MB ManifestArray<shape=(1, 433410...
    inundation_cells    (time, nele) int32 2MB ManifestArray<shape=(1, 433410...
Attributes: (12/14)
    title:                       SSCOFS
    institution:                 School for Marine Science and Technology
    source:                      FVCOM_4.4.7
    history:                     model started at: 01/11/2026   04:08
    references:                  http://fvcom.smast.umassd.edu, https://githu...
    Conventions:                 CF-1.0
    ...                          ...
    Tidal_Forcing:               Tidal Forcing Time Series Title: sscofs late...
    River_Forcing:               THERE ARE 38 RIVERS IN THIS MODEL.\nRIVER IN...
    GroundWater_Forcing:         GROUND WATER FORCING IS OFF!
    Surface_Heat_Forcing:        FVCOM variable surface heat forcing file:\nF...
    Surface_Wind_Forcing:        FVCOM variable surface Wind forcing:\nFILE N...
    Surface_PrecipEvap_Forcing:  SURFACE PRECIPITATION FORCING IS OFF

In [5]:
# Create the icechunk
vds_list = [
    open_virtual_dataset(url=u, parser=parser, registry=registry, loadable_variables=coords)
    for u in model_best_files
]
combined = xarray.concat(vds_list, dim="time", data_vars="minimal", coords="minimal",
                     compat="override", combine_attrs="override")

creds = icechunk.containers_credentials({bucket: icechunk.s3_credentials(anonymous=True)})


storage = icechunk.s3_storage(
    bucket=icechunk_bucket,
    prefix=icechunk_prefix,
    region="us-east-1",
    from_env=True,
)

config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=bucket,
        store=icechunk.s3_store(region="us-east-1", anonymous=True),
    )
)

repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=creds)
repo.save_config()

# Store the small time coordinate as one chunk, not one chunk per file,
# so the store stays quick to open as more days are added.
combined["time"].encoding["chunks"] = (len(combined.time),)

session = repo.writable_session("main")
combined.vz.to_icechunk(session.store)
session.commit("SSCOFS icechunk")

'4EF9HTB065NB8BJXM9SG'

In [6]:
#Open icechunk stored in s3

repo = icechunk.Repository.open(storage, authorize_virtual_chunk_access=creds)
session = repo.writable_session("main")
ds_ic = xarray.open_zarr(session.store, consolidated=False, chunks={})
ds_ic = xsg.grids.ugrid.assign_ugrid_topology(
    ds_ic, face_node_connectivity="nv",
    face_face_connectivity="nbe",
    start_index=1,
)

# The source science variables reference fvcom_mesh, but assign_ugrid_topology
# creates mesh. GridLook follows each variable's mesh attribute, so repair it.
for name in ds_ic.variables:
    if ds_ic[name].attrs.get("mesh") == "fvcom_mesh":
        ds_ic[name].attrs["mesh"] = "mesh"

# Inspect the assigned topology before committing.



# %%
root = zarr.open_group(session.store, mode="r+", use_consolidated=False)
root.attrs.update(dict(ds_ic.attrs))

for name, variable in ds_ic.variables.items():
    if name in root:
        root[name].attrs.update(dict(variable.attrs))
    else:
        # Write only new variables (such as the scalar mesh), preserving
        # existing virtual references without rewriting the science arrays.
        xarray.Dataset({name: variable}).to_zarr(
            session.store, mode="a", consolidated=False
        )

snapshot_id = session.commit("Add UGRID topology metadata")

In [7]:
ds_ic = xarray.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
ds_ic

<xarray.Dataset> Size: 589MB
Dimensions:             (four: 4, nele: 433410, node: 239734, three: 3,
                         time: 5, maxnode: 11, maxelem: 9, siglev: 11,
                         siglay: 10)
Coordinates:
    lonc                (nele) float32 2MB dask.array<chunksize=(433410,), meta=np.ndarray>
    latc                (nele) float32 2MB dask.array<chunksize=(433410,), meta=np.ndarray>
    lat                 (node) float32 959kB dask.array<chunksize=(239734,), meta=np.ndarray>
    lon                 (node) float32 959kB dask.array<chunksize=(239734,), meta=np.ndarray>
  * time                (time) datetime64[ns] 40B 2026-11-01T03:00:00 ... 202...
Dimensions without coordinates: four, nele, node, three, maxnode, maxelem,
                                siglev, siglay
Data variables: (12/44)
    a1u                 (four, nele) float32 7MB dask.array<chunksize=(4, 433410), meta=np.ndarray>
    a2u                 (four, nele) float32 7MB dask.array<chunksize=(4, 433410), meta=np.ndarray>
    art2                (node) float32 959kB dask.array<chunksize=(239734,), meta=np.ndarray>
    aw0                 (three, nele) float32 5MB dask.array<chunksize=(3, 433410), meta=np.ndarray>
    awy                 (three, nele) float32 5MB dask.array<chunksize=(3, 433410), meta=np.ndarray>
    iint                (time) int32 20B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                  ...
    wet_cells           (time, nele) int32 9MB dask.array<chunksize=(1, 433410), meta=np.ndarray>
    ww                  (time, siglay, nele) float32 87MB dask.array<chunksize=(1, 4, 144470), meta=np.ndarray>
    wet_nodes           (time, node) int32 5MB dask.array<chunksize=(1, 239734), meta=np.ndarray>
    zeta                (time, node) float32 5MB dask.array<chunksize=(1, 239734), meta=np.ndarray>
    uwind_speed         (time, nele) float32 9MB dask.array<chunksize=(1, 433410), meta=np.ndarray>
    yc                  (nele) float32 2MB dask.array<chunksize=(433410,), meta=np.ndarray>

In [8]:
%%time
bbox =  (-125.3, 44.29, -123.8, 45.11)
ds_sub = ds_ic.xsg.subset_bbox(bbox)
ds_sub

CPU times: user 790 ms, sys: 334 ms, total: 1.12 s
Wall time: 3.4 s


<xarray.Dataset> Size: 4MB
Dimensions:             (four: 4, nele: 2875, node: 1711, three: 3, time: 5,
                         maxnode: 11, maxelem: 9, siglev: 11, siglay: 10)
Coordinates:
    lonc                (nele) float32 12kB dask.array<chunksize=(2875,), meta=np.ndarray>
    latc                (nele) float32 12kB dask.array<chunksize=(2875,), meta=np.ndarray>
    lat                 (node) float32 7kB dask.array<chunksize=(1711,), meta=np.ndarray>
    lon                 (node) float32 7kB dask.array<chunksize=(1711,), meta=np.ndarray>
  * time                (time) datetime64[ns] 40B 2026-11-01T03:00:00 ... 202...
Dimensions without coordinates: four, nele, node, three, maxnode, maxelem,
                                siglev, siglay
Data variables: (12/44)
    a1u                 (four, nele) float32 46kB dask.array<chunksize=(4, 2875), meta=np.ndarray>
    a2u                 (four, nele) float32 46kB dask.array<chunksize=(4, 2875), meta=np.ndarray>
    art2                (node) float32 7kB dask.array<chunksize=(1711,), meta=np.ndarray>
    aw0                 (three, nele) float32 34kB dask.array<chunksize=(3, 2875), meta=np.ndarray>
    awy                 (three, nele) float32 34kB dask.array<chunksize=(3, 2875), meta=np.ndarray>
    iint                (time) int32 20B dask.array<chunksize=(1,), meta=np.ndarray>
    ...                  ...
    wet_cells           (time, nele) int32 58kB dask.array<chunksize=(1, 2875), meta=np.ndarray>
    ww                  (time, siglay, nele) float32 575kB dask.array<chunksize=(1, 4, 2875), meta=np.ndarray>
    wet_nodes           (time, node) int32 34kB dask.array<chunksize=(1, 1711), meta=np.ndarray>
    zeta                (time, node) float32 34kB dask.array<chunksize=(1, 1711), meta=np.ndarray>
    uwind_speed         (time, nele) float32 58kB dask.array<chunksize=(1, 2875), meta=np.ndarray>
    yc                  (nele) float32 12kB dask.array<chunksize=(2875,), meta=np.ndarray>